# f6_m02b_dice.ipynb

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M02b — Contrafactuales DiCE |

---

## 🎯 Qué hace

Genera explicaciones contrafactuales con DiCE (Microsoft): dado un alumno
predicho como abandono, ¿qué cambios mínimos en sus features harían que
el modelo predijera no-abandono?
Analiza 3 perfiles: abandono real (TP), falso positivo (FP) y zona de riesgo.

## 📋 Requisitos

- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/X_train_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/y_train.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`
- Entorno: `tfm_abandono` (dice-ml instalado)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/dice_contrafactuales.parquet` | Tabla de contrafactuales por perfil |
| `results/fase6/dice_resumen.png` | Gráfico comparativo original vs contrafactual |
| `docs/html/fase6/m02b_dice.html` | Informe HTML |

## 🔄 Flujo

```
X_train_prep + pipeline CatBoost
    ↓ DiCE: construir explainer con modelo scikit-learn wrapper
    ↓ Seleccionar 3 instancias (TP abandono, FP, zona riesgo)
    ↓ Generar contrafactuales (proximity + diversity)
    ↓ Tabla comparativa + gráfico
    → parquet + HTML
```

## ➡️ Siguiente

`f6_m03_fairness_indice.ipynb` → índice del grupo Fairness


In [1]:
# ============================================================================
# CELDA 1: IMPORTS Y RUTAS
# ============================================================================
# ROOT detection estándar TFM — sube niveles hasta encontrar src/
# ============================================================================

import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = DIR_DATA / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'DIR_DATA: {DIR_DATA}')
print(f'DIR_RESULTS: {DIR_RESULTS}')


ROOT: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_DATA: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado
DIR_RESULTS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6


In [2]:
# ============================================================================
# CELDA 2: IMPORTS
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import dice_ml
from dice_ml import Dice
from src.html.render import render_pagina_desde_fichero

plt.rcParams['figure.dpi'] = 120
print('Imports OK.')


Imports OK.


In [3]:
# ============================================================================
# CELDA 3: CARGAR DATOS Y MODELO
# ============================================================================
# DiCE necesita datos de entrenamiento para construir el explainer.
# Usamos el pipeline completo (no extraer named_steps) para predict_proba.
# ============================================================================

X_test_prep  = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
X_train_prep = pd.read_parquet(DIR_DATA / 'X_train_prep.parquet')
y_test       = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
y_train      = pd.read_parquet(DIR_DATA / 'y_train.parquet').squeeze()
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')

# Features numéricas — DiCE necesita saber cuáles puede modificar
features_num = X_test_prep.select_dtypes(include=[np.number]).columns.tolist()
features_cat = X_test_prep.select_dtypes(exclude=[np.number]).columns.tolist()

print(f'X_test_prep:  {X_test_prep.shape}')
print(f'X_train_prep: {X_train_prep.shape}')
print(f'Features num: {len(features_num)} | cat: {len(features_cat)}')


X_test_prep:  (6725, 19)
X_train_prep: (26896, 19)
Features num: 19 | cat: 0


In [4]:
# ============================================================================
# CELDA 4: SELECCIONAR INSTANCIAS REPRESENTATIVAS
# ============================================================================
# 3 perfiles para análisis contrafactual:
#   - Perfil A: abandono real predicho correctamente (True Positive)
#   - Perfil B: falso positivo (predicho abandono, no abandona)
#   - Perfil C: zona de riesgo (prob entre 0.45 y 0.55)
# ============================================================================

prob_test = pipeline_cat.predict_proba(X_test_prep)[:, 1]
pred_test = (prob_test >= 0.5).astype(int)
y_test_arr = y_test.values

# Perfil A — True Positive (abandono real, predicho correctamente)
mask_tp = (y_test_arr == 1) & (pred_test == 1)
idx_tp  = X_test_prep[mask_tp].index[0]

# Perfil B — False Positive (no abandona pero predicho como abandono)
mask_fp = (y_test_arr == 0) & (pred_test == 1)
idx_fp  = X_test_prep[mask_fp].index[0]

# Perfil C — Zona de riesgo (probabilidad entre 0.45 y 0.55)
mask_riesgo = (prob_test >= 0.45) & (prob_test <= 0.55)
idx_riesgo  = X_test_prep[mask_riesgo].index[0] if mask_riesgo.sum() > 0 else X_test_prep.index[0]

instancias = {
    'TP — Abandono real': idx_tp,
    'FP — Falso positivo': idx_fp,
    'Zona de riesgo': idx_riesgo,
}

print('Instancias seleccionadas:')
for nombre, idx in instancias.items():
    print(f'  {nombre}: idx={idx}, prob={prob_test[X_test_prep.index.get_loc(idx)]:.3f}')


Instancias seleccionadas:
  TP — Abandono real: idx=8609, prob=0.993
  FP — Falso positivo: idx=315, prob=0.534
  Zona de riesgo: idx=315, prob=0.534


In [5]:
# ============================================================================
# CELDA 5: CONSTRUIR EXPLAINER DiCE
# ============================================================================
# DiCE necesita un wrapper del modelo compatible con scikit-learn.
# Usamos 'sklearn' backend con el pipeline completo.
# Solo features numéricas son modificables (continuous).
# Las categóricas se mantienen fijas para contrafactuales realistas.
# ============================================================================

# Dataset DiCE con datos de entrenamiento
train_df = X_train_prep.copy()
train_df['abandono'] = y_train.values

d = dice_ml.Data(
    dataframe=train_df,
    continuous_features=features_num,
    outcome_name='abandono'
)

# Modelo wrapper — DiCE con backend sklearn
m = dice_ml.Model(model=pipeline_cat, backend='sklearn')

# Explainer con método random (más rápido que genético para diagnóstico)
exp = Dice(d, m, method='random')

print('DiCE explainer construido correctamente.')


DiCE explainer construido correctamente.


In [6]:
# ============================================================================
# CELDA 6: GENERAR CONTRAFACTUALES
# ============================================================================
# Método random — no acepta proximity_weight ni diversity_weight.
# Usamos solo los parámetros compatibles con todas las versiones de dice-ml.
# ============================================================================

resultados_dice = []

for nombre_perfil, idx in instancias.items():
    instancia = X_test_prep.loc[[idx]]

    try:
        dice_exp = exp.generate_counterfactuals(
            instancia,
            total_CFs=3,
            desired_class='opposite',
            verbose=False
        )
        cf_df = dice_exp.cf_examples_list[0].final_cfs_df
        if cf_df is not None and len(cf_df) > 0:
            cf_df = cf_df.drop(columns=['abandono'], errors='ignore')
            cf_df['perfil'] = nombre_perfil
            cf_df['tipo']   = 'contrafactual'
            orig = instancia.copy()
            orig['perfil'] = nombre_perfil
            orig['tipo']   = 'original'
            resultados_dice.append(pd.concat([orig, cf_df], ignore_index=True))
            print(f'  ✅ {nombre_perfil}: {len(cf_df)} contrafactuales generados')
        else:
            print(f'  ⚠️  {nombre_perfil}: sin contrafactuales válidos')
    except Exception as e:
        print(f'  ❌ {nombre_perfil}: error — {e}')

if resultados_dice:
    df_dice = pd.concat(resultados_dice, ignore_index=True)
    df_dice.to_parquet(DIR_RESULTS / 'dice_contrafactuales.parquet')
    print(f'Parquet guardado: {len(df_dice)} filas')
else:
    df_dice = pd.DataFrame()
    print('⚠️  Sin resultados — se genera HTML informativo')


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.79it/s]

100%|██████████| 1/1 [00:00<00:00,  3.78it/s]

  ✅ TP — Abandono real: 3 contrafactuales generados


  0%|          | 0/1 [00:00<?, ?it/s]

...

  ✅ FP — Falso positivo: 3 contrafactuales generados


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.11it/s]

100%|██████████| 1/1 [00:00<00:00,  3.05it/s]

  ✅ Zona de riesgo: 3 contrafactuales generados
Parquet guardado: 12 filas


In [7]:
# ============================================================================
# CELDA 7: GRÁFICO COMPARATIVO ORIGINAL vs CONTRAFACTUAL
# ============================================================================
# Para cada perfil muestra las features que cambian entre original y
# el primer contrafactual. Solo muestra features numéricas con delta != 0.
# ============================================================================

import base64

graficos_b64 = {}

for nombre_perfil, idx in instancias.items():
    subset = df_dice[df_dice['perfil'] == nombre_perfil] if len(df_dice) > 0 else pd.DataFrame()
    if len(subset) < 2:
        continue

    orig_row = subset[subset['tipo'] == 'original'][features_num].iloc[0]
    cf_row   = subset[subset['tipo'] == 'contrafactual'][features_num].iloc[0]

    # Features que cambian
    delta = (cf_row - orig_row).abs()
    top_features = delta.nlargest(8).index.tolist()
    if not top_features:
        continue

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(top_features))
    w = 0.35
    ax.bar(x - w/2, orig_row[top_features], w, label='Original', color='#e53e3e', alpha=0.8)
    ax.bar(x + w/2, cf_row[top_features],   w, label='Contrafactual', color='#38a169', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(top_features, rotation=30, ha='right', fontsize=9)
    ax.set_title(f'Contrafactual — {nombre_perfil}', fontsize=12)
    ax.legend(fontsize=9)
    plt.tight_layout()

    clave = nombre_perfil.replace(' ', '_').replace('—', '').strip('_')
    ruta_png = DIR_RESULTS / f'dice_{clave}.png'
    plt.savefig(ruta_png, dpi=120, bbox_inches='tight')
    plt.close()

    with open(ruta_png, 'rb') as f:
        graficos_b64[nombre_perfil] = base64.b64encode(f.read()).decode()
    print(f'  Gráfico guardado: {ruta_png.name}')

print('Gráficos listos.')


  Gráfico guardado: dice_TP__Abandono_real.png


  Gráfico guardado: dice_FP__Falso_positivo.png


  Gráfico guardado: dice_Zona_de_riesgo.png
Gráficos listos.


In [8]:
# ============================================================================
# CELDA 8: GENERAR HTML
# ============================================================================

def bloque_cf(nombre_perfil: str, idx, b64: str) -> str:
    prob = prob_test[X_test_prep.index.get_loc(idx)]
    img_html = (
        f'<img src="data:image/png;base64,{b64}" '
        f'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        if b64 else
        '<p style="color:#718096;font-size:13px">Contrafactual no generado para este perfil.</p>'
    )
    return (
        f'<div style="margin:28px 0;padding:20px;background:#f7fafc;border-radius:8px;'
        f'border-left:4px solid #3182ce">'
        f'<h3 style="color:#2d3748;font-size:15px;margin-bottom:6px">{nombre_perfil}</h3>'
        f'<p style="color:#718096;font-size:12px;margin-bottom:12px">'
        f'Probabilidad original de abandono: <strong>{prob:.3f}</strong></p>'
        f'{img_html}'
        f'</div>'
    )

bloques = ''
for nombre_perfil, idx in instancias.items():
    clave = nombre_perfil.replace(' ', '_').replace('—', '').strip('_')
    b64   = graficos_b64.get(nombre_perfil, '')
    bloques += bloque_cf(nombre_perfil, idx, b64)

n_cf = len(df_dice[df_dice['tipo'] == 'contrafactual']) if len(df_dice) > 0 else 0

contenido = (
    '<h2 style="color:#2d3748">Contrafactuales DiCE — ¿Qué cambiaría la predicción?</h2>'
    '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    'Los contrafactuales explican qué cambios mínimos en las características de un alumno '
    'harían que el modelo cambiara su predicción de abandono a no-abandono. '
    'Se analizan tres perfiles: un verdadero positivo (abandono real detectado), '
    'un falso positivo (alarma incorrecta) y un caso en zona de riesgo.'
    '</p>'
    f'<p style="color:#4a5568;font-size:13px;margin-top:8px">'
    f'Contrafactuales generados: <strong>{n_cf}</strong> | '
    f'Perfiles analizados: <strong>{len(instancias)}</strong>'
    f'</p>'
    + bloques +
    '<div style="margin-top:24px;padding:16px;background:#ebf8ff;'
    'border-left:4px solid #3182ce;border-radius:6px;font-size:13px;color:#2c5282">'
    '<strong>Interpretación:</strong> Las barras rojas muestran el valor original del alumno '
    'y las verdes el valor contrafactual que cambiaría la predicción. '
    'Features con mayor diferencia son las más accionables para intervención temprana.'
    '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m02b_dice.ipynb', contenido)
ruta_html = ROOT / 'docs' / 'html' / 'fase6' / 'm02b_dice.html'
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')


✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m02b_dice.html
